# 🔀 OCT Augmented Dataset Splitter

**Purpose:** Split the augmented OCT segmentation dataset into **train / val / test** splits without data leakage.

**Key principle:** All augmented crops that come from the **same original image** are kept in the **same split**. This prevents the model from seeing augmented versions of a test image during training.

---

### Filename convention (from augmentation notebook)

```
{original_stem}_t{nn}_r{row}_c{col}.png

Examples:
  patient001_t00_r+0_c+0.png     ← baseline crop (no shift)
  patient001_t01_r+12_c-8.png   ← translated crop 1
  patient001_t02_r-5_c+20.png   ← translated crop 2
```

The part **before** `_t` is the **original image ID**.

---

### Split ratios
| Split | Ratio |
|-------|-------|
| Train | 70%   |
| Val   | 15%   |
| Test  | 15%   |

---

### Output
Three text files, each listing the **full image paths** for that split:
- `train.txt`
- `val.txt`
- `test.txt`


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted.')

Mounted at /content/drive
✅ Google Drive mounted.


## Step 2 — Configuration

Set your paths and split ratios here. **Only edit this cell.**

> `OCT_DIR` must point to the folder produced by the augmentation notebook  
> (`/content/drive/MyDrive/FYP/data/augmented/crop_oct`).  
> The mask folder is used only for cross-checking that every image has a matching mask.


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
OCT_DIR   = "/content/drive/MyDrive/FYP/data/augmented/crop_oct"
MASK_DIR  = "/content/drive/MyDrive/FYP/data/augmented/crop_mask"
OUT_DIR   = "/content/drive/MyDrive/FYP/data/augmented"   # where txt files are saved

# ── Split ratios (must sum to 1.0) ─────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Reproducibility ────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── Sanity check ───────────────────────────────────────────────────────────
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, \
    "Split ratios must sum to 1.0"

print(f"OCT  dir : {OCT_DIR}")
print(f"Mask dir : {MASK_DIR}")
print(f"Output   : {OUT_DIR}")
print(f"Splits   : train={TRAIN_RATIO:.0%}  val={VAL_RATIO:.0%}  test={TEST_RATIO:.0%}")
print(f"Seed     : {RANDOM_SEED}")

OCT  dir : /content/drive/MyDrive/FYP/data/augmented/crop_oct
Mask dir : /content/drive/MyDrive/FYP/data/augmented/crop_mask
Output   : /content/drive/MyDrive/FYP/data/augmented
Splits   : train=70%  val=15%  test=15%
Seed     : 42


## Step 3 — Import Libraries

In [ ]:
import os
import random
import re
from pathlib import Path
from collections import defaultdict

print('✅ Libraries imported.')

✅ Libraries imported.


## Step 4 — Read Image Filenames & Group by Original ID

Each augmented filename has the form:
```
{original_stem}_t{nn}_r{row}_c{col}.png
```
We split on `_t` to recover the **original image ID**, then group all augmented crops under that ID.


In [ ]:
oct_dir  = Path(OCT_DIR)
mask_dir = Path(MASK_DIR)

# ── Collect all PNG files ───────────────────────────────────────────────────
all_files = sorted(oct_dir.glob('*.png'))

if len(all_files) == 0:
    raise FileNotFoundError(
        f"No .png files found in:\n  {OCT_DIR}\n"
        "Please check your OCT_DIR path."
    )

print(f"Found {len(all_files)} PNG files in OCT dir.")

# ── Group by original image ID ──────────────────────────────────────────────
# Filename format: {original_stem}_t{nn}_r{...}_c{...}.png
# Split on '_t' to get original stem — same approach used in the augmentation QC cell.
groups = defaultdict(list)   # original_id  →  [filename, ...]
unmatched = []

for fpath in all_files:
    stem = fpath.stem   # e.g.  patient001_t02_r+12_c-8
    match = re.match(r'^(.+?)_t\d+', stem)
    if match:
        orig_id = match.group(1)   # e.g.  patient001
        groups[orig_id].append(fpath.name)
    else:
        # Fallback: treat whole stem as original ID (e.g. files with no _t suffix)
        groups[stem].append(fpath.name)
        unmatched.append(fpath.name)

if unmatched:
    print(f"⚠️  {len(unmatched)} file(s) did not match the expected pattern "
          f"and were treated as standalone originals.")
    print("   First few:", unmatched[:5])

print(f"\n📂 Unique original image IDs : {len(groups)}")
print(f"🖼️  Total augmented files      : {len(all_files)}")

# ── Preview a few groups ────────────────────────────────────────────────────
print("\nSample groups (original_id → number of augmented files):")
for i, (orig_id, files) in enumerate(list(groups.items())[:5]):
    print(f"  {orig_id:40s}  →  {len(files)} file(s)")
if len(groups) > 5:
    print(f"  ... and {len(groups)-5} more groups.")

Found 5094 PNG files in OCT dir.

📂 Unique original image IDs : 3049
🖼️  Total augmented files      : 5094

Sample groups (original_id → number of augmented files):
  0                                         →  6 file(s)
  1000                                      →  1 file(s)
  1001                                      →  1 file(s)
  1002                                      →  1 file(s)
  1003                                      →  1 file(s)
  ... and 3044 more groups.


## Step 5 — Verify Mask Coverage

Every OCT crop must have a matching segmentation mask. This cell checks for missing masks and warns you before the split.


In [ ]:
missing_masks = []

for fpath in all_files:
    mask_path = mask_dir / fpath.name
    if not mask_path.exists():
        missing_masks.append(fpath.name)

if missing_masks:
    print(f"⚠️  {len(missing_masks)} OCT file(s) are missing a paired mask:")
    for m in missing_masks[:10]:
        print(f"   {m}")
    if len(missing_masks) > 10:
        print(f"   ... and {len(missing_masks)-10} more.")
    print("\nThese files will still be included in the split lists.")
    print("Fix missing masks before training.")
else:
    print(f"✅ All {len(all_files)} OCT files have a matching mask.")

✅ All 5094 OCT files have a matching mask.


## Step 6 — Split by Original Image Group

We shuffle the list of **original image IDs** (not individual files) with a fixed seed, then assign groups to splits based on the configured ratios.  

This guarantees **no data leakage**: all augmented crops from the same original image always land in the same split.


In [ ]:
random.seed(RANDOM_SEED)

# ── Shuffle original IDs ────────────────────────────────────────────────────
orig_ids = list(groups.keys())
random.shuffle(orig_ids)
n_total = len(orig_ids)

# ── Compute split boundaries ────────────────────────────────────────────────
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val   # absorb any rounding remainder into test

train_ids = orig_ids[:n_train]
val_ids   = orig_ids[n_train : n_train + n_val]
test_ids  = orig_ids[n_train + n_val :]

# ── Collect file paths per split ────────────────────────────────────────────
def collect_paths(id_list, groups, oct_dir):
    """Return sorted list of full OCT paths for the given original IDs."""
    paths = []
    for oid in id_list:
        for fname in groups[oid]:
            paths.append(str(Path(oct_dir) / fname))
    return sorted(paths)

train_paths = collect_paths(train_ids, groups, OCT_DIR)
val_paths   = collect_paths(val_ids,   groups, OCT_DIR)
test_paths  = collect_paths(test_ids,  groups, OCT_DIR)

print("Split complete (no data leakage — grouped by original image ID)")
print(f"\n{'Split':<8} {'Orig. IDs':>10} {'Aug. Files':>12} {'% of files':>12}")
print("-" * 46)
for name, ids, paths in [("train", train_ids, train_paths),
                          ("val",   val_ids,   val_paths),
                          ("test",  test_ids,  test_paths)]:
    pct = len(paths) / len(all_files) * 100
    print(f"{name:<8} {len(ids):>10} {len(paths):>12} {pct:>11.1f}%")
print("-" * 46)
print(f"{'TOTAL':<8} {n_total:>10} {len(all_files):>12} {'100.0':>11}%")

Split complete (no data leakage — grouped by original image ID)

Split     Orig. IDs   Aug. Files   % of files
----------------------------------------------
train          2134         3504        68.8%
val             457          832        16.3%
test            458          758        14.9%
----------------------------------------------
TOTAL          3049         5094       100.0%


## Step 7 — Save Split Files

Each `.txt` file contains **one file path per line** — the full path to the OCT image on Google Drive.

These files can be consumed directly by a PyTorch `Dataset` or any training script that reads a path list.


In [ ]:
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

splits = {
    'train': train_paths,
    'val':   val_paths,
    'test':  test_paths,
}

saved_files = {}
for split_name, paths in splits.items():
    txt_path = out_dir / f"{split_name}.txt"
    with open(txt_path, 'w') as f:
        f.write('\n'.join(paths) + '\n')
    saved_files[split_name] = txt_path
    print(f"✅ Saved {split_name:5s}.txt  ({len(paths)} lines)  →  {txt_path}")

print("\nAll split files saved.")

✅ Saved train.txt  (3504 lines)  →  /content/drive/MyDrive/FYP/data/augmented/train.txt
✅ Saved val  .txt  (832 lines)  →  /content/drive/MyDrive/FYP/data/augmented/val.txt
✅ Saved test .txt  (758 lines)  →  /content/drive/MyDrive/FYP/data/augmented/test.txt

All split files saved.


## Step 8 — Statistics Summary

In [ ]:
aug_per_orig = [len(v) for v in groups.values()]
avg_aug = sum(aug_per_orig) / len(aug_per_orig)
min_aug = min(aug_per_orig)
max_aug = max(aug_per_orig)

print("=" * 55)
print("          DATASET SPLIT STATISTICS")
print("=" * 55)
print(f"  Original image groups        : {n_total}")
print(f"  Total augmented files        : {len(all_files)}")
print(f"  Avg augmentations per image  : {avg_aug:.1f}")
print(f"  Min augmentations per image  : {min_aug}")
print(f"  Max augmentations per image  : {max_aug}")
print("-" * 55)
print(f"  TRAIN  — {len(train_ids):>4} orig. IDs  |  {len(train_paths):>5} aug. files  ({len(train_paths)/len(all_files)*100:.1f}%)")
print(f"  VAL    — {len(val_ids):>4} orig. IDs  |  {len(val_paths):>5} aug. files  ({len(val_paths)/len(all_files)*100:.1f}%)")
print(f"  TEST   — {len(test_ids):>4} orig. IDs  |  {len(test_paths):>5} aug. files  ({len(test_paths)/len(all_files)*100:.1f}%)")
print("=" * 55)
print(f"  Random seed : {RANDOM_SEED}")
print(f"  Output dir  : {OUT_DIR}")
print("=" * 55)

          DATASET SPLIT STATISTICS
  Original image groups        : 3049
  Total augmented files        : 5094
  Avg augmentations per image  : 1.7
  Min augmentations per image  : 1
  Max augmentations per image  : 6
-------------------------------------------------------
  TRAIN  — 2134 orig. IDs  |   3504 aug. files  (68.8%)
  VAL    —  457 orig. IDs  |    832 aug. files  (16.3%)
  TEST   —  458 orig. IDs  |    758 aug. files  (14.9%)
  Random seed : 42
  Output dir  : /content/drive/MyDrive/FYP/data/augmented


## Step 9 — Verify: No Overlap Between Splits

Double-check that no original image ID appears in more than one split (strict no-leakage proof).


In [ ]:
train_set = set(train_ids)
val_set   = set(val_ids)
test_set  = set(test_ids)

tv_overlap = train_set & val_set
tt_overlap = train_set & test_set
vt_overlap = val_set   & test_set

if tv_overlap or tt_overlap or vt_overlap:
    print("❌ DATA LEAKAGE DETECTED:")
    if tv_overlap: print(f"   Train ∩ Val  : {tv_overlap}")
    if tt_overlap: print(f"   Train ∩ Test : {tt_overlap}")
    if vt_overlap: print(f"   Val   ∩ Test : {vt_overlap}")
else:
    print("✅ No overlap between splits — dataset is leak-free!")
    print(f"   Train IDs : {len(train_set)}  |  Val IDs : {len(val_set)}  |  Test IDs : {len(test_set)}")

# Also verify total coverage
covered = train_set | val_set | test_set
all_ids_set = set(orig_ids)
if covered == all_ids_set:
    print("✅ All original IDs are assigned to exactly one split.")
else:
    missing = all_ids_set - covered
    print(f"⚠️  {len(missing)} ID(s) not assigned to any split: {list(missing)[:5]}")

✅ No overlap between splits — dataset is leak-free!
   Train IDs : 2134  |  Val IDs : 457  |  Test IDs : 458
✅ All original IDs are assigned to exactly one split.


## Step 10 — Show Sample Groups from Each Split

Preview a few original image IDs and their augmented files assigned to each split.


In [ ]:
N_SHOW = 3   # number of sample groups to show per split

for split_name, id_list in [('TRAIN', train_ids),
                              ('VAL',   val_ids),
                              ('TEST',  test_ids)]:
    print(f"\n{'─'*55}")
    print(f"  {split_name} — sample groups")
    print(f"{'─'*55}")
    sample = id_list[:N_SHOW]
    for orig_id in sample:
        files = sorted(groups[orig_id])
        print(f"  Original ID : {orig_id}  ({len(files)} augmented file(s))")
        for fname in files:
            print(f"    • {fname}")
    if len(id_list) > N_SHOW:
        print(f"  ... and {len(id_list) - N_SHOW} more original IDs in {split_name}.")

print(f"\n{'─'*55}")
print("Preview complete.")


───────────────────────────────────────────────────────
  TRAIN — sample groups
───────────────────────────────────────────────────────
  Original ID : 2509  (1 augmented file(s))
    • 2509_t00.png
  Original ID : 1534  (1 augmented file(s))
    • 1534_t00.png
  Original ID : 1032  (6 augmented file(s))
    • 1032_t00.png
    • 1032_t01_r+5_c+7.png
    • 1032_t02_r+14_c-3.png
    • 1032_t03_r-6_c-11.png
    • 1032_t04_r+10_c+3.png
    • 1032_t05_r-12_c+12.png
  ... and 2131 more original IDs in TRAIN.

───────────────────────────────────────────────────────
  VAL — sample groups
───────────────────────────────────────────────────────
  Original ID : 393  (1 augmented file(s))
    • 393_t00.png
  Original ID : 511  (1 augmented file(s))
    • 511_t00.png
  Original ID : 2913  (1 augmented file(s))
    • 2913_t00.png
  ... and 454 more original IDs in VAL.

───────────────────────────────────────────────────────
  TEST — sample groups
───────────────────────────────────────────────────

## Step 11 — Quick-Read Utility (Optional)

A helper function showing how to read the split `.txt` files back into Python lists — useful in your training notebook.


In [ ]:
def read_split_txt(txt_path):
    """Read a split .txt file and return a list of image paths."""
    with open(txt_path, 'r') as f:
        return [line.strip() for line in f if line.strip()]


# ── Example usage in your training notebook ─────────────────────────────────
# train_images = read_split_txt("/content/drive/MyDrive/FYP/data/augmented/train.txt")
# val_images   = read_split_txt("/content/drive/MyDrive/FYP/data/augmented/val.txt")
# test_images  = read_split_txt("/content/drive/MyDrive/FYP/data/augmented/test.txt")
#
# To get the matching mask path, just replace the OCT dir with the mask dir:
# mask_path = path.replace('/crop_oct/', '/crop_mask/')


# ── Verify read-back ────────────────────────────────────────────────────────
for split_name, txt_path in saved_files.items():
    lines = read_split_txt(txt_path)
    print(f"{split_name:5s}.txt  →  {len(lines)} paths read back correctly")

print("\n✅ Split files verified. Ready for training!")
print(f"\nNext step: load train.txt / val.txt / test.txt in your UNet fine-tuning notebook.")
print(f"Files are at: {OUT_DIR}")

train.txt  →  3504 paths read back correctly
val  .txt  →  832 paths read back correctly
test .txt  →  758 paths read back correctly

✅ Split files verified. Ready for training!

Next step: load train.txt / val.txt / test.txt in your UNet fine-tuning notebook.
Files are at: /content/drive/MyDrive/FYP/data/augmented


---

## ✅ Done

Your dataset is now split without data leakage. The three files:

```
/content/drive/MyDrive/FYP/data/augmented/train.txt
/content/drive/MyDrive/FYP/data/augmented/val.txt
/content/drive/MyDrive/FYP/data/augmented/test.txt
```

each contain **full paths** to the OCT images in that split. Use them in your `finetune_unet_roi_fixed.ipynb` like this:

```python
train_images = [line.strip() for line in open('train.txt')]
train_masks  = [p.replace('/crop_oct/', '/crop_mask/') for p in train_images]
```

**No files were copied or moved** — only lightweight text lists were created. 🎉
